# Práctica 3: Descenso de gradiente

**Fundamentos matemáticos del aprendizaje profundo** · Departamento de Matemática, FCEN-UBA

Material de referencia: capítulos 4 de las notas del curso. Convenciones: $y=\varphi(w\cdot x+b)$, $\sigma$ es la logística, $\eta$ la tasa de aprendizaje, $w_{ij}$ conecta la entrada $i$ con la salida $j$.

Cada práctica tiene una **Parte A** (ejercicios teóricos, para resolver en papel) y una **Parte B** (computacional). Las celdas marcadas `# TODO` son para completar.

## Parte A. Ejercicios teóricos

**A1.** Sea $f(x,y)=e^x\cos y$. Mostrar que $f$ es armónica ($f_{xx}+f_{yy}=0$), calcular $\|\nabla f\|$ y probar que $\nabla f=0$ no tiene soluciones; por lo tanto $f$ no tiene extremos locales. ¿Qué hace el descenso de gradiente sobre $f$?

**A2.** (Función cuadrática) Sea $f(x)=\tfrac12x^\top Ax-b^\top x$ con $A$ simétrica definida positiva, autovalores $0<\mu=\lambda_1\le\dots\le\lambda_n=L$. Mostrar que para $x_{k+1}=x_k-\eta\nabla f(x_k)$ vale
$$\|x_{k+1}-x^*\|\le\max_i|1-\eta\lambda_i|\;\|x_k-x^*\|.$$
Hallar el $\eta$ óptimo y la tasa $\frac{\kappa-1}{\kappa+1}$, $\kappa=L/\mu$.

**A3.** (Lema de descenso) Si $\nabla f$ es $L$-Lipschitz, mostrar $f(y)\le f(x)+\nabla f(x)\cdot(y-x)+\frac L2\|y-x\|^2$. Deducir que para $\eta\le 1/L$,
$f(x_{k+1})\le f(x_k)-\frac\eta2\|\nabla f(x_k)\|^2$.

**A4.** Con las hipótesis de A3 y $f$ acotada inferiormente, mostrar que $\min_{k\le K}\|\nabla f(x_k)\|^2\le \frac{2(f(x_0)-f_*)}{\eta (K+1)}$. Si además $f$ es convexa, $f(x_K)-f_*\le \frac{\|x_0-x^*\|^2}{2\eta K}$.

**A5.** Sea $f(x)=x^4$. Mostrar que el descenso de gradiente converge pero no linealmente. ¿Qué pasa con $\eta$ fijo si $|x_0|$ es grande?

**A6.** (Momentum) Escribir el método de la bola pesada $x_{k+1}=x_k-\eta\nabla f(x_k)+\beta(x_k-x_{k-1})$ como un sistema de primer orden y analizar su estabilidad en el caso $f(x)=\frac\lambda2x^2$.

## Parte B. Computacional

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
plt.rcParams['figure.figsize']=(6,4)
rng=np.random.default_rng(0)

### B1. Descenso de gradiente en una cuadrática mal condicionada
$f(x)=\frac12(x_1^2+\kappa x_2^2)$. Implementar GD y graficar trayectorias para $\eta\in\{0.1/\kappa,\,1/\kappa,\,1.9/\kappa,\,2.1/\kappa\}$ y $\kappa=1,10,50$.

In [ ]:
def gd(grad,x0,eta,n):
    xs=[np.array(x0,float)]
    for _ in range(n): xs.append(xs[-1]-eta*grad(xs[-1]))
    return np.array(xs)
kappa=10
grad=lambda x: np.array([x[0],kappa*x[1]])
g1,g2=np.meshgrid(np.linspace(-3,3,200),np.linspace(-1,1,200))
for eta in [0.05,0.1,0.19,0.21]:
    tr=gd(grad,[3,1],eta,30)
    plt.contour(g1,g2,0.5*(g1**2+kappa*g2**2),15,colors='lightgray')
    plt.plot(tr[:,0],tr[:,1],'.-'); plt.title(f'eta={eta}  (2/L={2/kappa})'); plt.ylim(-1.2,1.2); plt.show()

**B1.1** Medir la tasa empírica $\|x_{k+1}-x^*\|/\|x_k-x^*\|$ y compararla con $\max_i|1-\eta\lambda_i|$ para $\eta=1/L$ y para el $\eta$ óptimo de A2.

In [ ]:
# TODO

### B2. Convergencia sublineal: $x^4$
Correr GD sobre $f(x)=x^4$ y graficar $|x_k|$ en escala log-log. ¿Con qué velocidad decae? Compararlo con $|x_k|\sim k^{-1/2}$.

In [ ]:
f=lambda x: x**4
df=lambda x: 4*x**3
x=1.0; xs=[x]
for _ in range(2000):
    x=x-0.05*df(x); xs.append(x)
k=np.arange(1,len(xs))
plt.loglog(k,np.abs(xs[1:]),label='|x_k|'); plt.loglog(k,k**-0.5/np.sqrt(8*0.05),'--',label='(8 eta k)^{-1/2}')
plt.legend(); plt.show()

### B3. Momentum
Implementar la bola pesada y Nesterov. Comparar con GD en la cuadrática del B1 con $\kappa=100$ (número de iteraciones para llegar a error $10^{-6}$).

In [ ]:
# TODO

### B4. GD estocástico
Para $C(w,b)=\frac1N\sum_i\frac12(wx_i+b-y_i)^2$ con los datos de la Práctica 2, implementar GD, SGD (un dato por paso) y mini-batch. Graficar el costo vs. número de evaluaciones de gradiente de un dato. ¿Qué pasa con $\eta$ constante vs. $\eta_k\sim1/k$?

In [ ]:
N=200
x=rng.uniform(-1,1,N); y=2*x-1+0.3*rng.standard_normal(N)
def costo(p): return 0.5*np.mean((p[0]*x+p[1]-y)**2)
def grad_idx(p,idx):
    r=p[0]*x[idx]+p[1]-y[idx]
    return np.array([np.mean(r*x[idx]),np.mean(r)])
# TODO: GD (idx = todo), SGD (idx=[i]), mini-batch de tamano 16

### B5. Mínimos locales y puntos silla
Para $f(x,y)=x^4-2x^2+y^2$ (dos mínimos) y $g(x,y)=x^2-y^2$ (silla), correr GD desde distintos puntos iniciales y graficar a qué punto converge cada uno (cuencas de atracción).

In [ ]:
# TODO